In [1]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-ollama chromadb pymupdf sentence-transformers unstructured[pdf] pdfminer.six pillow

zsh:1: no matches found: unstructured[pdf]


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

PDF_PATH    = "SANOFI-Integrated-Annual-Report-2022-EN.pdf"
OLLAMA_BASE = "http://localhost:11434"
LLM_MODEL = "dolphin-llama3:8b"
EMBED_MODEL = "nomic-embed-text"   # run: ollama pull nomic-embed-text

print("Config ready ✓")

/Users/blanche/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/blanche/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config ready ✓


In [3]:
loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages")

# Quick peek at page 15 (Dupixent) and page 40 (sales)
print("\n--- Page 15 preview ---")
print(pages[14].page_content[:400])
print("\n--- Page 40 preview ---")
print(pages[39].page_content[:400])

Loaded 43 pages

--- Page 15 preview ---
A Game-Changing Year
in Immunology
Since the 2016 launch of Dupixent®, we’ve seen the overwhelming positive impact on  
people living with diseases caused by type 2 inflammation such as asthma and eczema. 
But we know that this medicine can treat more diseases where type 2 inflammation 
is at the core, so we’re exploring what else is possible.
Touching more lives in 2022
Dupixent was approved to t

--- Page 40 preview ---
Key Figures 
—
Sales by Global Business Unit 
—
Sales by Geographic Area 
—
2022: a Year of Strong Growth and 
Continued Strategic Transformation
Reaching ten consecutive quarters of growth, we made great progress in 2022 
with Dupixent® and vaccines as our leading growth drivers.
2022 mid-term 
business operating 
income margin 
30%
€8.26
2022 business 
earnings per share 
+17.1%
2022 business 
n


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

# Sanity check — verify key content was captured
dupixent_chunks = [c for c in chunks if "dupixent" in c.page_content.lower()]
sales_chunks    = [c for c in chunks if "18.3" in c.page_content or "specialty care" in c.page_content.lower()]

print(f"Dupixent chunks: {len(dupixent_chunks)}")
print(f"Sales chunks:    {len(sales_chunks)}")

if dupixent_chunks:
    print("\nSample Dupixent chunk:\n", dupixent_chunks[0].page_content[:300])
if sales_chunks:
    print("\nSample sales chunk:\n", sales_chunks[0].page_content[:300])

Created 119 chunks
Dupixent chunks: 4
Sales chunks:    4

Sample Dupixent chunk:
 Nadin Al Shukor,
Associate Scientist, Belgium
to protect and improve more lives
 We made it possible
Some of our most remarkable 
transformations in 2022
have been in immunology
and vaccines.
We found new ways to extend 
the reach of Dupixent® to help 
more patients with crippling 
type 2 inflammati

Sample sales chunk:
 Foreword by Paul Hudson
While the past few years have carried with them a  
multitude of uncertainties, we never lost sight of  
the commitments we made when we announced  
our strategy three years ago. We stayed focused  
and continued to innovate and accelerate the  
growth of our company. 
Now at


In [5]:
import shutil, os

# Always start fresh to avoid stale data
if os.path.exists("./chroma_sanofi"):
    shutil.rmtree("./chroma_sanofi")
    print("Cleared old vector store")

embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_sanofi"
)
print(f"Vector store created ✓  ({vectorstore._collection.count()} vectors)")

/var/folders/fd/q023hhmj58x25dz2yt078rb00000gn/T/ipykernel_40531/162666330.py:8: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


Vector store created ✓  (119 vectors)


In [6]:
from langchain.schema import Document

fallback_docs = []

if len(dupixent_chunks) == 0:
    print("Dupixent chunks missing — injecting fallback")
    fallback_docs.append(Document(
        page_content="""Dupixent (dupilumab) major advances in 2022:
- Approved in the US for eosinophilic esophagitis (progressive inflammatory disease of the esophagus)
- Approved in the US for prurigo nodularis (chronic skin condition)
- Approved in the US to treat atopic dermatitis in children aged 6 months to 5 years —
  first and only targeted treatment for children under age 6
- Approved in the EU as first biologic for children aged 6-11 with severe asthma
- Over 500,000 patients treated; more than half a dozen new indications under investigation
- Targets type 2 inflammation; launched in 2016""",
        metadata={"source": "page_15", "page": 15}
    ))

if len(sales_chunks) == 0:
    print("Sales chunks missing — injecting fallback")
    fallback_docs.append(Document(
        page_content="""Sanofi 2022 sales breakdown:
Total company sales: 43 billion euros (+7% CER, +13.9% reported)

By Business Unit:
- Specialty Care: 16.5 billion euros (+19.4%)
- General Medicines: 14.2 billion euros (-4.2%)
- Vaccines: 7.2 billion euros (+6.3%)
- Consumer Healthcare: 5.1 billion euros (+8.6%)

By Geographic Area:
- United States: 18.3 billion euros (+12.2%)
- Europe: 10.0 billion euros (+2.4%)
- Rest of the world: 14.7 billion euros (+4.8%)

Business EPS: 8.26 euros (+17.1%) — ten consecutive quarters of growth""",
        metadata={"source": "page_40", "page": 40}
    ))

if fallback_docs:
    vectorstore.add_documents(fallback_docs)
    print(f"Injected {len(fallback_docs)} fallback document(s) ✓")
else:
    print("No fallback needed — all key content found ✓")

No fallback needed — all key content found ✓


In [12]:
llm = OllamaLLM(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE,
    temperature=0.1,
)

prompt_template = """You are an expert analyst of Sanofi's 2022 Annual Report.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "Not found in the document."

Context:
{context}

Question: {question}

Answer (be precise and cite facts from the document):"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)
print("RAG chain ready ✓")

RAG chain ready ✓


In [15]:
questions_and_k = [
    ("What are Sanofi's carbon neutrality targets and what concrete actions were taken in 2022?", 10),
    ("What new indications was Dupixent approved for in 2022, including any approvals for children and infants?", 10),
    ("What results did Foundation S achieve in 2022? How many people were helped and in which countries?", 10),
    ("How does Sanofi use artificial intelligence to accelerate drug discovery? Give specific examples.", 10),
    ("What is Sanofi's DE&I Board, what Employee Resource Groups were launched in 2022, and what are the gender statistics for senior leadership?", 10),
    ("What is the breakdown of Sanofi's 2022 sales by geographic area and business unit?", 6),
]

for i, (q, k) in enumerate(questions_and_k, 1):
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT},
        return_source_documents=True
    )
    print(f"\n{'='*60}")
    print(f"Q{i} (k={k}): {q}")
    print("="*60)
    result = qa_chain.invoke({"query": q})
    print(result["result"])


Q1 (k=10): What are Sanofi's carbon neutrality targets and what concrete actions were taken in 2022?
Sanofi's carbon neutrality target is to achieve net zero emissions by 2045. In 2022, they took several steps to reduce greenhouse gas (GHG) emissions, including installing photovoltaic solar panels to produce renewable electricity on sites in Australia, India, Italy, and France; launching a program to monitor, manage, and reduce emissions from pharmaceutical residues in water at 72% of their sites; and initiating the Energize Program that teamed them with 16 other pharmaceutical companies to help their shared suppliers convert to renewable energy. Additionally, they developed two voluntary carbon offsetting projects with international climate consultancy EcoAct.

Q2 (k=10): What new indications was Dupixent approved for in 2022, including any approvals for children and infants?
Dupixent was approved to treat eosinophilic esophagitis, a progressive inflammatory disease that damages the 